In [27]:
from leagueScripts import PlayTypeLeagueAverage
from nba_api.stats.library.parameters import PlayType
from nba_api.stats.endpoints import SynergyPlayTypes
import pandas as pd

playtypes_df = []
for playtype in PlayTypeLeagueAverage.get_playtype_classes_names():
    _playtype_df = SynergyPlayTypes(play_type_nullable=getattr(PlayType, playtype), type_grouping_nullable='offensive', player_or_team_abbreviation='P').synergy_play_type.get_data_frame()
    if not _playtype_df.empty:
        playtypes_df.append(_playtype_df)
df = pd.concat(playtypes_df)
df = df[['PLAYER_NAME', 'TEAM_NAME', 'PLAY_TYPE', 'PTS', 'POSS', 'POSS_PCT']]

# Define a custom aggregation function
def custom_agg(group):
    # Calculate the combined team names
    combined_team_names = ', '.join(group['TEAM_NAME'].unique())
    
    # Calculate the sum of PTS and POSS
    pts_sum = group['PTS'].sum()
    poss_sum = group['POSS'].sum()
    
    # Calculate the correct POSS_PCT
    poss_pct_numerator = poss_sum
    poss_pct_denominator = (group['POSS'] / group['POSS_PCT']).sum()
    poss_pct = poss_pct_numerator / poss_pct_denominator if poss_pct_denominator != 0 else 0
    
    # Return as a Series
    return pd.Series({
        'TEAM_NAME': combined_team_names,
        'PTS': pts_sum,
        'POSS': poss_sum,
        'POSS_PCT': poss_pct
    })

# Apply the custom aggregation for players with multiple teams
df = df.groupby(['PLAYER_NAME', 'PLAY_TYPE']).apply(custom_agg, include_groups=False)
df['PPP'] = df['PTS'] / df['POSS']
# Calculate percentiles
df['PERCENTILE_FREQ%'] = df.groupby('PLAY_TYPE')['POSS_PCT'].rank(pct=True) * 100

df

TEAM_NAME  PTS  POSS  POSS_PCT  \
PLAYER_NAME     PLAY_TYPE                                                  
AJ Green        Handoff             Milwaukee Bucks   12    15     0.074   
                Isolation           Milwaukee Bucks   26    22     0.108   
                PRBallHandler       Milwaukee Bucks   71    71     0.348   
                Spotup              Milwaukee Bucks   66    72     0.353   
Aaron Gordon    Handoff              Denver Nuggets   44    57     0.072   
...                                             ...  ...   ...       ...   
Zion Williamson Handoff        New Orleans Pelicans   39    39     0.056   
                OffScreen      New Orleans Pelicans   43    21     0.030   
                PRBallHandler  New Orleans Pelicans  215   235     0.338   
                PRRollMan      New Orleans Pelicans   31    30     0.043   
                Spotup         New Orleans Pelicans  297   267     0.384   

                                    PPP  PERCENTILE_FREQ%  
PLAYER_NAME     PLAY_TYPE                                  
AJ Green        Handoff        0.800000         48.767606  
                Isolation      1.181818         80.303030  
                PRBallHandler  1.000000         33.615819  
                Spotup         0.916667         64.814815  
Aaron Gordon    Handoff        0.771930         43.133803  
...                                 ...               ...  
Zion Williamson Handoff        1.000000         19.542254  
                OffScreen      2.047619         10.256410  
                PRBallHandler  0.914894         29.943503  
                PRRollMan      1.033333         69.721116  
                Spotup         1.112360         78.347578  

[2145 rows x 6 columns]

In [28]:
# Pivot the dataframe
df_pivot = df.pivot_table(index=['PLAYER_NAME'], 
                          columns='PLAY_TYPE', 
                          values=['PPP', 'POSS_PCT', 'PERCENTILE_FREQ%'], 
                          aggfunc='first', 
                          fill_value=0)

# Flatten the columns
# df_pivot.columns = [f'{i}_{j}' for i, j in df_pivot.columns]
df_pivot = df_pivot.rename(columns={'PLAYER_NAME_': 'PLAYER_NAME', 'TEAM_NAME_': 'TEAM_NAME'})

df_ppp_for_plot = df_pivot['PPP']
df_frequency_for_plot = df_pivot['POSS_PCT']
df_percentile_frequency_for_plot = df_pivot['PERCENTILE_FREQ%']
metrics = df_ppp_for_plot.columns.tolist()

In [29]:
df_ppp_for_plot

PLAY_TYPE,Handoff,Isolation,OffScreen,PRBallHandler,PRRollMan,Postup,Spotup
PLAYER_NAME,,,,,,,
AJ Green,0.800000,1.181818,0.000000,1.000000,0.000000,0.000000,0.916667
Aaron Gordon,0.771930,0.914286,0.000000,0.000000,0.000000,1.024390,0.923913
Aaron Holiday,1.137255,0.722222,1.272727,0.935000,1.166667,1.148148,1.051546
Aaron Nesmith,0.970588,0.000000,0.952381,1.060241,1.350000,1.125000,0.000000
Aaron Wiggins,0.857143,0.695652,0.956522,0.000000,1.000000,0.857143,0.000000
...,...,...,...,...,...,...,...
Zach Collins,1.133333,1.000000,0.562500,0.955789,1.345455,1.043478,1.110294
Zach LaVine,1.307692,0.764706,0.388889,0.898734,0.000000,1.125000,1.163462
Zavier Simpson,0.000000,0.000000,0.000000,0.809524,0.000000,0.000000,0.000000


In [30]:
df_frequency_for_plot

PLAY_TYPE,Handoff,Isolation,OffScreen,PRBallHandler,PRRollMan,Postup,Spotup
PLAYER_NAME,,,,,,,
AJ Green,0.074,0.108,0.000,0.348,0.000,0.000,0.353
Aaron Gordon,0.072,0.089,0.000,0.000,0.000,0.052,0.350
Aaron Holiday,0.115,0.081,0.049,0.449,0.027,0.061,0.218
Aaron Nesmith,0.079,0.000,0.073,0.386,0.023,0.093,0.000
Aaron Wiggins,0.073,0.096,0.048,0.000,0.025,0.044,0.000
...,...,...,...,...,...,...,...
Zach Collins,0.019,0.071,0.020,0.594,0.069,0.058,0.170
Zach LaVine,0.051,0.067,0.071,0.311,0.000,0.063,0.409
Zavier Simpson,0.000,0.000,0.000,0.429,0.000,0.000,0.000


In [31]:
df_percentile_frequency_for_plot

PLAY_TYPE,Handoff,Isolation,OffScreen,PRBallHandler,PRRollMan,Postup,Spotup
PLAYER_NAME,,,,,,,
AJ Green,48.767606,80.303030,0.000000,33.615819,0.000000,0.000000,64.814815
Aaron Gordon,43.133803,60.606061,0.000000,0.000000,0.000000,44.536424,64.102564
Aaron Holiday,95.598592,48.030303,37.728938,72.598870,28.087649,62.417219,14.957265
Aaron Nesmith,58.626761,0.000000,84.432234,50.706215,11.553785,96.523179,0.000000
Aaron Wiggins,45.774648,68.787879,34.249084,0.000000,16.932271,25.000000,0.000000
...,...,...,...,...,...,...,...
Zach Collins,5.281690,30.000000,7.142857,89.830508,92.031873,57.615894,5.982906
Zach LaVine,14.964789,23.484848,81.868132,16.525424,0.000000,66.556291,86.324786
Zavier Simpson,0.000000,0.000000,0.000000,66.101695,0.000000,0.000000,0.000000


In [32]:
from leagueScripts import NBALeague

league_object = NBALeague.get_cached_league_object()

top_scorers = sorted(league_object.players_on_teams_objects_list, key=lambda x: x.stats_df['PTS'].sum().item() if not x.stats_df.empty else 0, reverse=True)[:100]
players_to_plot = [player.player_info['DISPLAY_FIRST_LAST'].item() for player in top_scorers]
title = f"{', '.join(players_to_plot) if len(players_to_plot) <5 else 'Players'} play type stats"

In [33]:
import numpy as np
import plotly.graph_objects as go

# Create a new Figure
fig = go.Figure()

# Add averages
fig.add_trace(go.Scatterpolar(
        r=df_ppp_for_plot.replace(0, np.NaN).mean(),
        theta=metrics,
        name="Average",
        line=dict(color='white'),  # Set the color for the average trace
        showlegend=False  # Make the Average trace always visible and not part of the legend
    ))

# Plot each player in a separate chart
for player_name in players_to_plot:
    player_ppp_df = df_ppp_for_plot.loc[player_name]
    player_frequency_df = df_frequency_for_plot.loc[player_name]
    player_percentile_frequency_df = df_percentile_frequency_for_plot.loc[player_name]

    fig.add_trace(go.Scatterpolar(
        r=player_ppp_df.values,
        theta=metrics,
        fill='toself',
        name=player_name,
        marker=dict(
            size=player_percentile_frequency_df.values
        ),
        customdata=player_frequency_df.values * 100,  # Adding custom data
        hovertemplate='<b>%{theta}</b><br>PPP: %{r}<br>Frequency: %{customdata}%<extra></extra>'
    
    ))

# Update layout with increased width and height
fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 2]
        )),
    showlegend=True,
    title=title,
    width=1500,
    height=1000,
)

# Show the plot
fig.show()
